# METADATA TABLES CREATION

## 1. metadata.tables
- Static registry of logical tables.
- Central registry of all logical tables, source systems, and their ingestion configuration (mapping + architecture layers)

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.tables (
    table_id            INT,
    table_name          STRING,
    source_system       STRING,        -- sqlserver / csv / json / api
    source_schema       STRING,        -- dbo (null for files)
    source_table        STRING,        -- table name (null for files)
    source_path         STRING,        -- file path (null for sqlserver)
    target_layer        STRING,        -- silver/gold
    bronze_schema       STRING,        -- bronze
    silver_schema       STRING,        -- silver
    gold_schema         STRING,        -- gold
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
)
USING DELTA;

## 2. metadata.table_parameters
Stores configurable ETL parameters per table (load type, primary key, watermark column, etc.) to drive dynamic processing.
> load_type :
- MERGE → update + insert
- APPEND → insert only (keep full history)
- FULL → full reload (truncate and reload everything)

Example:
- customers → MERGE (update existing customers + add new ones)
- transactions → APPEND (keep all transaction history)
- branches → FULL (reload all branch data).

> watermark_column (incremental column):
- Used for incremental ingestion (load only recent changes). 
- updated_at
- txn_timestamp
- processed_timestamp 


In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA;

## 3. metadata.table_watermarks
- Stores last successful watermark per table
- Tracks the last successfully processed watermark for incremental data loading to avoid reprocessing existing records and improve pipeline performance.

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.table_watermarks (
    table_id                INT,
    last_watermark_value    STRING,     -- flexible type storage
    last_updated_at         TIMESTAMP,
    last_run_id             BIGINT
)
USING DELTA
PARTITIONED BY (table_id);

## 4. metadata.pipeline_runs
Stores execution logs for each pipeline run for monitoring, auditing, error tracking, and data lineage

In [0]:
CREATE TABLE IF NOT EXISTS banking.metadata.pipeline_runs (
    run_id              BIGINT,
    table_id            INT,
    layer               STRING,        -- Bronze / Silver / Gold
    start_time          TIMESTAMP,
    end_time            TIMESTAMP,
    status              STRING,        -- SUCCESS / FAILED
    number_of_records     BIGINT,
    error_message       STRING
)
USING DELTA
PARTITIONED BY (table_id);